# 33 · Regression, checked against statsmodels

## What this checks

**Regression** fits a model that links an outcome to an exposure, while adjusting for other factors. This check
covers release 0.9.0. Two kinds of model are checked:

- **Logistic regression,** for an outcome that is yes or no, such as being a case.
- **Poisson regression,** for an outcome that is a count, such as visits per day.

Each model gives a **coefficient** for every term, its effect on the model's scale. Each coefficient comes with a
**standard error**, a measure of how uncertain it is, and an interval. Fieldwork fits the models in its own engine,
written in Rust. Does it give the same coefficients, standard errors and intervals as statsmodels, for the same
declared model?

The two models are the two published ones the widget was built to reproduce:

- **Salmonella, Germany 2021.** Cases against two groups of controls, on eating Galia melon in April. Adjusted for age
  group (seven groups) and sex. The intervals are likelihood-ratio intervals, as published. These intervals are found
  by asking how far the coefficient can move before the model fits clearly worse. The rows are published under CC BY
  4.0, so they travel with the data.
- **New York City, 2016 to 2025.** Daily children's asthma emergency visits, on ozone. Adjusted for fine particles
  (PM2.5), nitrogen dioxide (NO2), a smooth curve for temperature (a spline) and holidays. The days are compared only
  within 840 groups, called strata. The standard errors are HAC errors, which allow for nearby days being alike. The
  counts belong to the city health department and are **not** passed on. Instead the check downloads the authors'
  package from Zenodo, about 9 MB, and checks its digest. That can be slow.

## How it is checked

We fit each model again with **statsmodels**, a Python statistics library by other authors. It uses its own fitting
method. The temperature curve is built by **patsy**, another Python library. statsmodels works out the HAC errors
itself. Nothing is shared with Fieldwork except the declared model.

Here the strata are fitted as one indicator column for each stratum. Fieldwork uses a conditional model instead. The
two routes give the same slopes, so they reach the same estimate by different paths.

statsmodels does not give likelihood-ratio intervals for these models. So we find each limit ourselves. We refit
statsmodels' model with the coefficient held fixed, and search for where the fit has worsened by the right amount.
The arithmetic of the search is this check's; the fits are statsmodels'.

This notebook runs in CPython only. The New York case reads its data from Zenodo in pieces, by range requests, which
the browser kernel cannot make. In the browser, the notebook says so and skips its later cells.

## What should happen

1. **Coefficients** agree within a relative difference of 0.000001, for every reported term.
2. **Standard errors** from the model agree within 0.000001.
3. **Likelihood-ratio limits** agree within 0.0001. Each is found by a search on both sides, so a looser margin is
   fair.
4. **HAC standard errors** agree within 0.000001 with statsmodels' Newey-West errors at 14 lags, with no small-sample
   correction. And the ozone coefficient and its HAC standard error equal the authors' archived result in R, within
   0.000000001.
5. **Rows:** both sides use the same number.

These criteria were fixed before the check was run.

**Two changes were made after the first run, on 5 November 2026, and are recorded as such.**

- **A tighter fit.** statsmodels is now fitted to a tolerance of 0.000000000001. At its default it stopped the
  Salmonella model one step early. Its standard error for Galia melon then differed from Fieldwork's by 0.000004.
  Tightened, the two agree to 15 digits.
- **Separated terms.** A term whose standard error is above 1,000 is called **separated**. No finite estimate exists
  for it. Two programs stop at different points while chasing it. Such a term is compared only on whether both find it
  separated.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas", "numpy", "scipy", "statsmodels", "patsy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
try:
    from checks.check_33_regression import run, rel
except ImportError as missing:
    report = None
    print("Not runnable here:", missing)
else:
    report = run()
    print("agrees:", report["agrees"])
    print()
    for c in report["comparisons"]:
        compared = [t for t in c["terms"] if not t.get("separated")]
        worst = max(max(rel(*t["coefficient"]), rel(*t["se"])) for t in compared)
        print(f"  {'OK ' if c['agrees'] else 'DIFF'} {c['fixture']:<10} {c['errors']:<8} {c['rows']['fieldwork']:>5} rows  "
              f"worst relative difference {worst:.1e}")
        print(f"       {c['note']}")

## The result on record

First recorded on 5 November 2026, against release 0.2.0. Rerun on 26 November 2026, against release 0.9.0:
**Agrees**.

Five comparisons agree:

- **Salmonella:** cases against the usual controls (142 rows) and against panel controls (113 rows), with
  likelihood-ratio and Wald intervals. The worst relative difference is 1.3 × 10⁻¹⁰. The published odds ratios, 96
  (15 to 2,067) and 18 (4.6 to 101), are recomputed.
- **New York City:** 3,643 rows and 840 strata. HAC and model-based errors agree within 1.1 × 10⁻¹². The authors' R
  result is reproduced.

The check also made **a finding against Fieldwork**, explained below.

Running the cell above repeats the check on the same data.

## The estimates a reader checks

The cell below shows each exposure's ratio and interval, beside statsmodels' and the published figure.

In [ ]:
if report is None:
    print("Skipped: this check is not runnable in this kernel (see above).")
else:
    import math
    for c in report["comparisons"]:
        first = c["terms"][0]
        b, b_sm = first["coefficient"]
        line = f"{c['fixture']:<10} {c['errors']:<8} {first['term']:<16} Fieldwork {math.exp(b):.4f}  statsmodels {math.exp(b_sm):.4f}"
        if "profile" in first:
            (lo, hi), (lo2, hi2) = first["profile"]["fieldwork"], first["profile"]["here"]
            line += f"   interval {lo:.2f}-{hi:.0f} against {lo2:.2f}-{hi2:.0f}"
        print(line)
        if "authorsArchivedR" in c:
            a = c["authorsArchivedR"]
            print(f"{'':>10} authors' R: coefficient {a['coefficient'][1]}  HAC SE {a['se'][1]}  agrees {a['agrees']}")

## A finding against Fieldwork: separation

Against the panel controls, the age group 80 and over is separated. Its people are either all cases or all controls.
So that group's odds ratio has **no finite estimate**: the fit keeps improving as the coefficient grows. Both programs
stop somewhere along the way, with standard errors of 29,232 and 1,596,022. Both agree it is separated.

**But Fieldwork reports it as an ordinary odds ratio, with no warning.** It shows 1.27 × 10⁸, from 0.016 to
2.8 × 10¹⁰. This check records that, and does not excuse it.

In [ ]:
if report is None:
    print("Skipped: this check is not runnable in this kernel (see above).")
else:
    for c in report["comparisons"]:
        for t in c["terms"]:
            if t.get("separated"):
                r = t["fieldworkReported"]
                print(f"{c['note'][:44]}...: {t['term']}, standard errors {t['se'][0]:.0f} (Fieldwork) and {t['se'][1]:.0f} (statsmodels)")
                print(f"  Fieldwork reported {r['ratio']:.3g} ({r['lower']:.3g} to {r['upper']:.3g})")

## What this does not show

- **That either model is well specified,** or that an adjusted estimate shows a cause.
- **The browser kernel.** This check runs in CPython only.
- **Offsets, overdispersed errors and exposures with more categories.** Neither example uses them.
- **Separated terms,** beyond the fact that both sides find them separated. Fieldwork reports them as ordinary odds
  ratios with no warning, as the panel case's 80-and-over group shows.

The cell below prints these limits as the check records them, with each data set's release.

In [ ]:
if report is None:
    print("Skipped: this check is not runnable in this kernel (see above).")
else:
    for item in report["notEstablished"]:
        print("-", item)
    print()
    for f in report["fixtures"]:
        print("fixture    :", f["id"], f["sha256"][:16], "· Fieldwork", f["application"]["version"], "· release", f["operation"]["release"])
    for k in ("statsmodels", "patsy", "pandas", "numpy"):
        print(f"{k:<11}:", report["environment"][k])
    print("runtime    :", report["environment"]["runtime"])